# 02 — Data Cleaning: CRMLS Sold Listings → AVM modeling table

**Input:** raw monthly CSVs in `data/`  
**Output:** `data/processed/sfr_clean.parquet` (one row per single-family sale) and `docs/cleaning_log.csv`  
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

All logic lives in `src/cleaning.py` so the same rules run from the notebook or the command line
(`python src/cleaning.py`). This notebook runs it, shows what each rule did, and sanity-checks the result.
Thresholds come from the EDA in `01_eda.ipynb`.

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cleaning

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Rules

| Step | Rule | Why |
|---|---|---|
| Population | `PropertyType == Residential` and `PropertySubType == SingleFamilyResidence`, state CA | AVM scope |
| Duplicates | One row per `ListingKey`, keep latest `ContractStatusChangeDate` | Same sale re-recorded with a corrected CloseDate |
| Target | Drop `ClosePrice` < $10k | Placeholder / error prices |
| Target | Drop close/list ratio outside [0.5, 2] | Catches extra/missing-zero typos (e.g. $970M on a $975k list) |
| Size | Drop `LivingArea` missing or outside 300–20,000 sq ft | Key feature; needed for price/sq ft check |
| Size | Drop price per sq ft outside $50–$5,000 | Remaining size/price mismatches |
| Features | Null impossible values (beds/baths ≤ 0 or > 20, YearBuilt < 1850 or in the future, negative DOM/parking, lots > 1,000 acres, coords outside CA) | Keep the sale, let the model impute the feature |
| Features | Fill `LotSizeSquareFeet` from `LotSizeAcres`, drop acres | One lot-size column |
| Flags | `latfilled` → `coords_imputed`; missing `BasementYN`/`WaterfrontYN` → False | Those two flags only ever record True |
| Columns | Drop PII, agent/office fields, IDs, constant and empty columns | Not property features; buyer-side info is only known after the sale |
| Derived | `close_month`, `log_close_price`, `property_age` | Split key, target, feature |

Thresholds are in `cleaning.RULES`:

In [ ]:
cleaning.RULES

## 3. Run

In [ ]:
raw = cleaning.load_raw("../data")
clean, log = cleaning.clean(raw)
log

In [ ]:
sfr_start = log.loc[log["step"] == "not Residential / SingleFamilyResidence", "rows_after"].item()
print(f"Single-family sales before cleaning: {sfr_start:,}")
print(f"After cleaning:                      {len(clean):,}  ({len(clean) / sfr_start:.1%} kept)")
print(f"Columns: {raw.shape[1]} raw -> {clean.shape[1]} clean")

## 4. Sanity checks on the clean table

In [ ]:
assert clean["ListingKey"].is_unique
assert clean["ClosePrice"].ge(cleaning.RULES["min_close_price"]).all()
assert clean["LivingArea"].between(*cleaning.RULES["living_area_sqft"]).all()
assert not set(cleaning.DROP_COLS) & set(clean.columns)
print("All checks passed")

In [ ]:
clean[["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt",
       "property_age", "LotSizeSquareFeet", "DaysOnMarket", "AssociationFee"]].describe().T

In [ ]:
missing = (clean.isna().mean() * 100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("% missing")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(clean["log_close_price"], bins=100)
axes[0].set_title("log(ClosePrice)")
axes[1].hist(clean["ClosePrice"] / clean["LivingArea"], bins=100)
axes[1].set_title("Price per sq ft")
clean.groupby("close_month").size().plot(kind="bar", ax=axes[2], title="Sales per month")
axes[2].tick_params(axis="x", labelsize=6)
plt.tight_layout(); plt.show()

## 5. Save

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
clean.to_parquet("../data/processed/sfr_clean.parquet", index=False)
log.to_csv("../docs/cleaning_log.csv", index=False)
print("Saved", clean.shape)

## 6. Notes for modeling

- **Target:** `log_close_price`. Report errors back in dollars.
- **Leakage watch:** `ListPrice`, `OriginalListPrice`, `DaysOnMarket`, `PurchaseContractDate` and `ContractStatusChangeDate` are kept but may not be allowed as features. Team decision pending.
- **Split:** by `close_month`. Latest month (2026-04) = test, second-latest = validation, the rest = train.
- **High-cardinality text** (`City`, `PostalCode`, `MLSAreaMajor`, school names, `SubdivisionName`) needs encoding; `Flooring` and `Levels` are comma-separated lists.